In [29]:
import cv2
import numpy as np
from pathlib import Path

In [33]:
def replace_background(
    video_path,
    background_path,
    output_path,
    history=500,
    varThreshold=24,
    detectShadows=False
):
    video_path = Path(video_path)
    background_path = Path(background_path)
    output_path = Path(output_path)

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    bg = cv2.imread(str(background_path))
    if bg is None:
        raise RuntimeError(f"Cannot open background image: {background_path}")
    bg = cv2.resize(bg, (width, height))

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(str(output_path), fourcc, fps, (width, height))

    subtractor = cv2.createBackgroundSubtractorMOG2(
        history=history,
        varThreshold=varThreshold,
        detectShadows=detectShadows
    )

    # Small kernel for removing noise
    kernel_open = np.ones((3, 3), np.uint8)

    # Larger kernels help recover missing hand/arm parts
    kernel_close = np.ones((7, 7), np.uint8)
    kernel_dilate = np.ones((5, 5), np.uint8)

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        fg_mask = subtractor.apply(frame)
        _, fg_mask = cv2.threshold(fg_mask, 200, 255, cv2.THRESH_BINARY)

        # Remove small noise
        fg_mask = cv2.morphologyEx(fg_mask, cv2.MORPH_OPEN, kernel_open, iterations=1)

        # Fill small gaps inside body/arms
        fg_mask = cv2.morphologyEx(fg_mask, cv2.MORPH_CLOSE, kernel_close, iterations=2)

        # Slightly expand the person to avoid losing hand parts
        fg_mask = cv2.dilate(fg_mask, kernel_dilate, iterations=1)

        # Soft edges for smoother composition
        fg_mask = cv2.GaussianBlur(fg_mask, (7, 7), 0)

        alpha = fg_mask.astype(np.float32) / 255.0
        alpha = alpha[:, :, None]

        frame_f = frame.astype(np.float32)
        bg_f = bg.astype(np.float32)

        result = frame_f * alpha + bg_f * (1.0 - alpha)
        result = np.clip(result, 0, 255).astype(np.uint8)

        writer.write(result)

    cap.release()
    writer.release()

In [31]:
video_path = "/content/denis.mp4"
background_path = "/content/imgg.png"
output_path = "output.mp4"

replace_background(video_path, background_path, output_path)
print("Saved:", output_path)

Saved: output.mp4
